# Head Trajectories Experiment Runner

This notebook is the complete notebook workflow for running **one or many** Head Trajectories experiments from setup to saved outputs.

It is meant to be the single distributable notebook for local or cloud use. You should be able to change a small configuration block, rerun the notebook, and get:

- probe datasets
- checkpoints
- probing results
- single-run summaries
- multi-seed profile summaries

The notebook is profile-driven. You choose experiment profiles and seeds; the repository code provides the actual training, probing, and analysis logic.

## What This Notebook Does

Running this notebook end to end will:

1. find or clone the repository
2. install any lightweight missing dependencies
3. let you define a run matrix of profiles and seeds
4. build or load the shared probe dataset for each profile
5. train all requested runs sequentially
6. probe all saved checkpoints sequentially
7. analyze each run
8. optionally compute profile-level multi-seed summaries

### Artifact layout

All artifacts are written under:

`ARTIFACT_ROOT / PROFILE_NAME / seed<SEED>`

Shared probe datasets live once per profile, and profile-level aggregate summaries live under:

`ARTIFACT_ROOT / PROFILE_NAME / aggregate`

### How to use this notebook

- Edit the configuration cell only.
- Define the profiles and seeds you want in `RUN_SPECS`.
- Run the notebook from top to bottom.
- For interrupted cloud sessions, keep `RESET_RUN = False` so existing artifacts are reused.

In [ ]:
# Repository bootstrap
from pathlib import Path
import importlib.util
import os
import subprocess
import sys


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'model').exists():
            return candidate
    return None


ROOT = find_repo_root(Path.cwd())
if ROOT is None:
    ROOT = Path.cwd() / 'head-trajectories'
    if not ROOT.exists():
        subprocess.run(
            ['git', 'clone', 'https://github.com/abderahmane-ai/head-trajectories.git', str(ROOT)],
            check=True,
        )

repo_str = str(ROOT.resolve())
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
os.chdir(ROOT)
print('ROOT =', ROOT.resolve())

missing = [
    pkg for pkg in ['datasets', 'tiktoken', 'scipy', 'matplotlib']
    if importlib.util.find_spec(pkg) is None
]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
    print('Installed:', missing)
else:
    print('Dependencies already available.')

## Choose Profiles and Seeds

Profiles define the full experimental setup:
- dataset
- model size
- training length
- probe sizes
- artifact naming

### Default profiles

- `wikitext103_15m_preliminary`  
  Faster representative pilot.

- `openwebtext_15m_main`  
  Main 15M OpenWebText experiment.

- `openwebtext_6m_ablation`  
  Smaller OpenWebText ablation for scale comparisons.

### Run matrix idea

Instead of configuring a single run, this notebook accepts a list of run specs. Each run spec contains:
- one profile
- one or more seeds

So you can request things like:
- one pilot run
- three main seeds
- one ablation run

all from the same notebook.

In [ ]:
from pprint import pprint

from experiments import (
    BatchRunSpec,
    get_profile,
    list_profiles,
    normalize_run_specs,
    resolve_artifacts,
    run_experiment_batch,
)

ARTIFACT_ROOT = ROOT / 'artifacts'
DEVICE = 'auto'
RESET_RUN = False
REBUILD_PROBE = False
RUN_TRAIN = True
RUN_PROBE = True
RUN_ANALYSIS = True
RUN_PROFILE_AGGREGATE = True

RUN_SPECS = [
    {'profile': 'wikitext103_15m_preliminary', 'seeds': [42]},
    # {'profile': 'openwebtext_15m_main', 'seeds': [42, 123, 777]},
    # {'profile': 'openwebtext_6m_ablation', 'seeds': [42]},
]

print('Available profiles:')
for profile in list_profiles():
    print(f'- {profile.name}: {profile.description}')

NORMALIZED_RUN_SPECS = normalize_run_specs(RUN_SPECS)

print('\nRequested run matrix:')
for spec in NORMALIZED_RUN_SPECS:
    profile = get_profile(spec.profile_name)
    pprint({
        'profile': profile.name,
        'description': profile.description,
        'seeds': list(spec.seeds),
        'dataset_name': profile.dataset_name,
        'dataset_config': profile.dataset_config,
        'dataset_family': profile.dataset_family,
        'n_layers': profile.model_config.n_layers,
        'n_heads': profile.model_config.n_heads,
        'd_model': profile.model_config.d_model,
        'd_ffn': profile.model_config.d_ffn,
        'total_steps': profile.total_steps,
        'batch_size': profile.batch_size,
        'block_size': profile.block_size,
        'artifact_root': str((ARTIFACT_ROOT / profile.name).resolve()),
    })
    print('-' * 80)

## Step 1: Validate the Planned Runs

This cell expands the run matrix into the exact profile/seed runs that will be executed.

Use it to confirm:
- the profiles are correct
- the seeds are correct
- the artifact root is what you expect

If you want to launch many runs sequentially, this is the cell to inspect before starting long training jobs.

In [ ]:
PLANNED_RUNS = []
for spec in NORMALIZED_RUN_SPECS:
    for seed in spec.seeds:
        paths = resolve_artifacts(spec.profile_name, seed=seed, artifact_root=ARTIFACT_ROOT)
        PLANNED_RUNS.append({
            'profile': spec.profile_name,
            'seed': seed,
            'probe_path': str(paths.probe_path.resolve()),
            'checkpoint_dir': str(paths.ckpt_dir.resolve()),
            'results_path': str(paths.results_path.resolve()),
            'summary_path': str(paths.summary_path.resolve()),
        })

print(f'Planned concrete runs: {len(PLANNED_RUNS)}')
for item in PLANNED_RUNS:
    pprint(item)

## Step 2: Run the Full Batch

This is the main execution cell.

For each requested profile and seed, it will run:
- probe dataset build/load
- training
- probing
- single-run analysis

If `RUN_PROFILE_AGGREGATE = True`, it will also compute a multi-seed aggregate summary for each profile after all of that profile's seeds are finished.

### Main control flags

- `RESET_RUN = True` starts each seed fresh
- `REBUILD_PROBE = True` rebuilds the shared probe dataset for each profile
- `RUN_TRAIN`, `RUN_PROBE`, `RUN_ANALYSIS` let you skip stages when resuming or reusing existing artifacts

In [ ]:
BATCH_RESULTS = run_experiment_batch(
    run_specs=NORMALIZED_RUN_SPECS,
    artifact_root=ARTIFACT_ROOT,
    device=DEVICE,
    reset_run=RESET_RUN,
    rebuild_probe=REBUILD_PROBE,
    run_train=RUN_TRAIN,
    run_probe=RUN_PROBE,
    run_analysis=RUN_ANALYSIS,
    run_profile_aggregate=RUN_PROFILE_AGGREGATE,
)

print('\nBatch complete.')
print('Executed runs   :', len(BATCH_RESULTS['runs']))
print('Profile summaries:', len(BATCH_RESULTS['aggregates']))

## Step 3: Inspect Single-Run Outputs and Figures

After the batch finishes, each requested run has its own saved analysis bundle.

For each run, the notebook now saves and exposes:
- the single-run summary
- onset information
- stability information
- phase-transition information
- the full figure bundle

The code cell below prints each run manifest and displays any saved figures inline.

In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path

for run_manifest in BATCH_RESULTS['runs']:
    display(Markdown(f"### Run: `{run_manifest['profile']['name']}` / seed `{run_manifest['seed']}`"))
    pprint(run_manifest)

    analysis = run_manifest.get('analysis', {})
    figures = analysis.get('figures', {})
    for label, figure_path in figures.items():
        path = Path(figure_path)
        if path.exists():
            display(Markdown(f"**{label.replace('_', ' ').title()}**"))
            display(Image(filename=str(path)))
    print('=' * 100)

## Step 4: Inspect Profile-Level Multi-Seed Summaries

If you requested multiple seeds for the same profile and left `RUN_PROFILE_AGGREGATE = True`, the batch runner also writes a profile-level aggregate summary.

This summary is useful for questions like:
- what is the average onset pattern across seeds?
- what do final fractions look like across runs?
- where is the aggregate timeline figure saved?

The next cell prints those summaries and shows the aggregate timeline figure when available.

In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path

if BATCH_RESULTS['aggregates']:
    for aggregate_summary in BATCH_RESULTS['aggregates']:
        display(Markdown(f"### Aggregate: `{aggregate_summary['profile']}`"))
        pprint(aggregate_summary)
        figure_path = aggregate_summary.get('timeline_figure')
        if figure_path and Path(figure_path).exists():
            display(Image(filename=str(Path(figure_path))))
        print('=' * 100)
else:
    print('No aggregate summaries were produced.')

## Final Artifact Summary

Use this final cell to print a compact inventory of the batch.

This is the cell to check before:
- stopping a cloud session
- sharing outputs with collaborators
- resuming work later
- deciding whether the runs finished cleanly

In [ ]:
import json

batch_manifest = {
    'artifact_root': str(ARTIFACT_ROOT.resolve()),
    'run_specs': BATCH_RESULTS['run_specs'],
    'runs': [run['artifacts'] for run in BATCH_RESULTS['runs']],
    'aggregates': BATCH_RESULTS['aggregates'],
}

batch_manifest_path = ARTIFACT_ROOT / 'batch_manifest.json'
batch_manifest_path.write_text(json.dumps(batch_manifest, indent=2), encoding='utf-8')

pprint(batch_manifest)
print('\nBatch manifest available at:', batch_manifest_path.resolve())